# BPI Challenge 2014: service restoration and change-related workload

The objective is to reduce avoidable service disruption and administrative transfers while retaining the specialist work needed to restore service. The incident activity history supports a process analysis. Interaction, incident and change case extracts add context and business questions, but are not complete event histories.

The analysis follows [PM²](https://vdaalst.com/publications/p824.pdf): Planning, Extraction, Data Processing, Mining and Analysis, Evaluation, and Process Improvement and Support. Each stage leaves a journal entry and an evidence question. Scores, priorities and sensitivity calculations come from WISE; this notebook selects questions and presents their results. No numerical finding is assumed before the source log has been analysed.

The owner describes anonymised service-management case extracts plus an incident activity log and asks how changes affect later service workload. [Incident dataset and source description](https://research.tue.nl/en/datasets/bpi-challenge-2014-incident-details/). This notebook treats the activity log as the observed sequence and the other three files as context extracts.

**Planning question.** Do repeated transfers, vendor-wait observations or reopening signals identify preventable coordination problems? Are those signals concentrated in particular priorities, incident categories or service components? A relationship between a change and an incident is an investigation lead; it does not prove the change caused the incident.

In [ ]:
import os
from dataclasses import replace
from pathlib import Path
from IPython.display import display, Markdown
import pandas as pd
import matplotlib.pyplot as plt
import wise

if wise.__version__ != "1.0.1":
    raise RuntimeError("Select the environment with the frozen 1.0.1 package.")

from bpic_analysis import (
    challenge, summary, norm_table, criterion_summary, activity_summary,
    context_profiles, priorities, plot_overview, case_evidence,
    threshold_sensitivity, view_agreement, integrity, journal, save_outputs, explanations,
)

DATA_ROOT = os.environ.get("WISE_PAPER_DATA")
EXPORT_RESULTS = False

## 1. Extraction: identify the population and observation boundary

Read each source from the supplied data folder. The summary records actual source populations and assessment status. Retain every observation of selected cases, including records outside a selected start-month cohort. A context extract without an event sequence is identified separately; it is not assigned a process score.

In [ ]:
runs = challenge(2014, data_folder=DATA_ROOT)
display(summary(runs))
scored_runs = {name: run for name, run in runs.items() if run.get("result") is not None}
display(journal(runs))

## 2. Data processing: check the meaning of observed activities

An activity update is not a measure of staff effort. A reassignment can be an appropriate referral, a vendor transfer can be necessary, and repeated pending observations do not prove separate waiting episodes. The incident extract’s resolution fields can supplement the history; sparse `Resolved` activities do not mean that cases were never resolved. Source timestamp ties need a declared ordering rule, and a many-to-one context join must not multiply activity rows.

The tables below distinguish event occurrences from cases with an observation. Counts are descriptive evidence; they are not effort, root causes or success rates.

In [ ]:
ACTIVITY_ANCHORS = ['Open', 'Assignment', 'Reassignment', 'External Vendor Reassignment', 'Pending vendor', 'Reopen', 'Resolved', 'Closed']
for name, run in scored_runs.items():
    display(Markdown(f"### {name}"))
    display(activity_summary(run, labels=ACTIVITY_ANCHORS))
    display(activity_summary(run).head(25))

## 3. Translate the business goal into a declared norm

The exploratory activity norm focuses on assignment timing, reassignment intensity, reopening observations and vendor-pending observations. Its cut points are review prompts, not service-level agreements. Interaction first-call resolution, verified incident closure, planned change-window overruns and required approval evidence need the corresponding case fields and policy interpretation. These questions are listed separately; a case extract must not be turned into invented lifecycle events to make them scoreable.

Read the criterion, scope, thresholds and view weights before interpreting a score. For `Singularity`, `k` is the allowed count and `K` is the additional count range over which severity increases; `K` is not an upper allowed count. Native lag matches the first response at or after the first activation, allowing equal timestamps. Pending questions are shown so that an unsupported business requirement cannot disappear behind a convincing chart.

In [ ]:
for name, run in scored_runs.items():
    display(Markdown(f"### {name}"))
    display(norm_table(run))
    display(run["specification"].get("pending_checks", []))

## 4. Establish the denominator and evidence coverage

Inspect unmatched incident identifiers, timestamp validity, context uniqueness and available resolution evidence before comparing service components. A missing activity endpoint can make an interval unavailable even when a case extract records a resolution date. Keep this distinction visible. Never replace unknown evidence with a compliant score.

`evaluated_cases` and `in_scope_cases` show different denominators. Mean severity refers to usable evidence; an unavailable observation is not a zero violation. A high mean severity with narrow coverage deserves a different response from a widespread, well-observed problem.

The library renormalises the score over usable evidence. A high score cannot certify adherence to the full norm when some evidence is unavailable. Criterion coverage is the unweighted fraction of in-scope cases that can be evaluated for that criterion. The case witness also shows each view’s weighted evidence coverage. These are different measures, and neither certifies that the source contains every real process event.

In [ ]:
criterion_tables = {}
for name, run in scored_runs.items():
    criterion_tables[name] = criterion_summary(run)
    display(Markdown(f"### {name}"))
    display(criterion_tables[name])

## 5. Mining and analysis: compare stakeholder priorities

The coordination and rework signals may elevate a component under the balanced or control-and-quality views, while the timeliness view may emphasise assignment intervals. Check that these leads are supported by the recorded activity vocabulary. A label absent from the source is not evidence that every incident violated the corresponding rule.

Priority combines scored case volume with a deficit in conformance score relative to the scored reference population. A higher conformance score means closer adherence to the declared norm; higher criterion severity means more deviation. The first rows show where to investigate, not which employee or process is at fault. An unranked or below-reference slice can still contain important individual cases.

In [ ]:
view_backlogs = {}
for name, run in scored_runs.items():
    for view in run["result"].views:
        backlog = priorities(run, view=view, gamma=0)
        view_backlogs[(name, view)] = backlog
        display(Markdown(f"### {name} — {view}"))
        display(backlog.table.head(10))

## 6. Visualise severity, evidence and investigation order

The overview uses computed library results. Read the severity and coverage panels together, then compare their message with the priority list. A frequent activity is not automatically an important failure; a rare signal should retain access to its supporting case.

In [ ]:
for name, run in scored_runs.items():
    display(Markdown(f"### {name}"))
    for figure in plot_overview(run):
        display(figure)
        plt.close(figure)
    table = criterion_tables[name]
    if not table.empty:
        figure, axis = plt.subplots(figsize=(8, 5), constrained_layout=True)
        plotted = table.dropna(subset=["evidence_coverage", "mean_severity"])
        for row in plotted.itertuples(index=False):
            axis.scatter(row.evidence_coverage, row.mean_severity, s=85,
                         label=row.criterion_id)
        axis.legend(title="Criterion", loc="center left", bbox_to_anchor=(1, 0.5))
        axis.set(xlim=(-0.03, 1.08), ylim=(-0.03, 1.08),
                 xlabel="Evidence coverage within business scope",
                 ylabel="Mean severity within usable evidence",
                 title="Does a severe signal also have broad usable evidence?")
        display(figure)
        plt.close(figure)

## 7. Explain the signed layer drivers

Read signed layer contributions next to the raw priority list. A slice with many reassignments may still have shorter initial assignment intervals than the reference population. Investigate the transfer mechanism rather than treating the total score as a verdict on a team. Legitimate specialist routing can coexist with preventable information loss.

Positive contributions increase the aggregate gap; negative contributions offset it. WISE keeps the signs and clips only the final sum. Do not remove the offsets to make a problem look larger.

In [ ]:
for (name, view), backlog in view_backlogs.items():
    explanation = explanations(scored_runs[name], backlog)
    display(Markdown(f"### {name} — {view}"))
    display(explanation["contributions"].reindex(backlog.table.head(10).index))
    display(explanation["drivers"].reindex(backlog.table.head(10).index))

## 8. Test thresholds, preferences and small-slice sensitivity

Ask whether a component remains a lead when the reassignment severity width changes while the allowed count stays fixed. If a vendor-related lead is stable, inspect whether its records represent repeated updates or distinct returns. Small slices can dominate a list without representing broad service impact; compare the declared shrinkage scenarios and keep the case counts visible.

These are declared scenarios, not estimates of an optimal policy. The timing scenario scales both the allowed lag and its severity width. The occurrence scenario keeps the allowed count `k` fixed and scales the additional severity width `K`; it does not halve or double the allowed count. The overlap measure concerns positive-priority shortlist membership and is undefined when both lists are empty. Low agreement is a prompt to explain the business trade-off.

In [ ]:
for name, run in scored_runs.items():
    display(Markdown(f"### {name}"))
    display(threshold_sensitivity(run, factors=(0.5, 1, 2)))
    display(view_agreement(run, k=10))
    view = run["result"].views[0]
    reference = priorities(run, view=view, gamma=0)
    for gamma in (5, 20, 100):
        adjusted = priorities(run, view=view, gamma=gamma)
        display({"dataset": name, "view": view, "gamma": gamma,
                 "positive_top10_overlap": wise.top_k_overlap(reference, adjusted, k=10)})
        display(adjusted.table.head(10))

## 9. Compare complete cases in meaningful context

Joined priority, category and affected service component are retrospective incident-extract snapshots repeated across that incident’s activity rows. A consistent joined value does not prove the classification was stable historically or was known at incident opening. These comparisons describe the extraction’s recorded classifications; they do not establish historical case mix or an escalation cause. A changing attribute needs a declared context rule, not a silent first-value classification. Case-start cohorts retain all later activity. The last observed month and observed-span band describe extract boundaries, not confirmed closure. Comparisons across linked changes require identity and observation-window checks before making a workload claim.

A start month selects cases rather than splitting their events. Derived first/last observed months use UTC after timestamp conversion; they are not verified business-local calendar months. A business-calendar analysis needs the source timezone and its calendar rules. `last_observed_month` and `observed_span_band` describe observation boundaries. They are not verified finish-month or completion measures. A context field that changes within a case is not silently treated as a stable business attribute in the comparisons below.

In [ ]:
CONTEXT_KEYS = ['Priority', 'Category', 'Service Component WBS (aff)', 'case_start_month', 'observed_span_band']
for name, run in scored_runs.items():
    display(Markdown(f"### {name}"))
    display(context_profiles(run, attributes=CONTEXT_KEYS))
    varying = run["log"].varying_case_attributes
    for attribute in CONTEXT_KEYS:
        if attribute not in run["result"].cases.columns:
            continue
        if varying.get(attribute, 0):
            display({"attribute": attribute, "comparison": "Context changes within cases; review the classification rule first"})
            continue
        display(Markdown(f"#### Whole-case grouping: {attribute}"))
        display(priorities(run, by=attribute, gamma=20).table.head(10))

## 10. Test how missing evidence changes the aggregation

Flat and layer-balanced scoring answer slightly different aggregation questions when evidence availability varies. Rescore the same complete cases through the library. Compare the resulting investigation order using identical slice keys and view; do not interpret a change in order as a newly observed process event.

In [ ]:
for name, run in scored_runs.items():
    view = run["result"].views[0]
    reference = priorities(run, view=view, gamma=0)
    by = list(reference.spec.by)
    for mode in ("flat", "layer_balanced"):
        rescored = replace(wise.score(run["log"], run["norm"], mode=mode), cases=run["result"].cases)
        backlog = wise.prioritize(rescored, by=by, view=view, gamma=0)
        display({"dataset": name, "view": view, "mode": mode,
                 "positive_top10_overlap": wise.top_k_overlap(reference, backlog, k=10)})
        display(backlog.table.head(10))

## 11. Evaluation: inspect the full case behind a lead

Inspect an incident highlighted for reassignment intensity. Review its full activity sequence and stable joined incident context. Look for a transfer back to an earlier team, what information was available at the handoff and whether a vendor or specialist referral was justified. The event label alone does not provide a transfer reason; record that gap before proposing a routing change.

The witness retains its source sequence and whole-case context. Compare the library evaluation with the matching observations and note what a missing endpoint means. The default witness is an investigation example, not a representative sample or proof of causation. To examine another case, pass its identifier to `case_evidence` without filtering away other events.

In [ ]:
case_witnesses = {}
for name, run in scored_runs.items():
    witness = case_evidence(run)
    case_witnesses[name] = witness
    display(Markdown(f"### Full source case: {name}"))
    for label, value in witness.items():
        display(Markdown(f"**{label}**"))
        display(value)

### Place the case evidence on a timeline

This figure uses event timestamps returned by the library for a complete recorded case selected by the first diagnostic criterion. Named lanes highlight norm activities; the grey lane keeps the remaining recorded activities visible. Coincident points can overlap and establish no order. Undated events remain in the source table and cannot be placed on the time axis. The selected case is an investigation example, not a representative sample; return to its full source witness before interpreting an apparent gap.

In [ ]:
from bpic_analysis import plot_case_timeline
for identifier, run in runs.items():
    if "result" not in run:
        continue
    figure = plot_case_timeline(run)
    display(figure)
    plt.close(figure)

## 12. Validate the result and decide what it supports

Use a service-component lead to formulate a limited test: does a complete diagnostic handoff reduce preventable returns without delaying appropriate escalation? For a change-related workload hypothesis, request release scope, timing and a comparable unaffected group. A linked incident count alone cannot identify a causal release effect or prove that a CAB approval was missing.

The integrity check records norm round-trip consistency, score decomposition, assessed case-index agreement and the retained event count. It does not independently certify extraction completeness, historical field validity or coverage of the real process. A failed check or incomplete binding limits the conclusion even when a plot looks convincing. Keep unsupported checks and cases without usable evidence in the journal.

### Baseline findings to carry into service review

The activity extract contains 466,737 events across 46,616 incidents. Interaction, incident and change extracts supply additional context, rather than three more scored event histories.

- Recorded reassignment intensity has mean severity 0.135 under the allowance of one observation and four-observation severity width; explicit reopening has mean severity 0.025 under zero allowed observations and width two. These are graded diagnostic values, not percentages of faulty incidents. Reassignment is the stronger review signal under these chosen cut points, but a specialist or vendor referral can be necessary.
- The first open-to-assignment interval is usable for 38,660 incidents (82.9%). There are 7,935 missing matched assignment responses and 21 missing open activations. Mean severity 0.032 therefore describes the usable interval evidence, rather than every incident’s assignment speed.
- Joined category, priority and service-component fields support retrospective comparisons. Their repetition across joined activity rows cannot establish the initial priority, historical ownership or the effect of a particular change. A change-impact claim still needs a validated affected-component link, a release date and a comparable observation window.

The feasible next step is to inspect reassignment-heavy incidents for repeated transfers without a diagnostic handoff. Trial a receiving-team acceptance rule only where that mechanism is confirmed; retain the specialist and emergency routes needed for resolution.

In [ ]:
for name, run in scored_runs.items():
    display(Markdown(f"### {name}"))
    display(integrity(run))
display(journal(runs))

## 13. Process improvement and support: choose a feasible pilot

The proposals below are conditional on the case review. They change the business process, not the software package. Select one supported mechanism, a responsible owner and comparable cases. Agree the outcome and quality safeguard before starting. An improved WISE score alone is not a business benefit.

In [ ]:
process_improvements = pd.DataFrame([{'Process issue to confirm': 'Incidents return because the receiving team lacks diagnostic information', 'Pilot change': 'Require a short diagnostic handoff and explicit acceptance by the receiving team for one service component', 'Owner': 'Service operations lead', 'Success measure': 'Fewer preventable returns and shorter verified restoration intervals in comparable priority/category cohorts', 'Safeguard': 'No delayed specialist escalation or deterioration in service availability'}, {'Process issue to confirm': 'Vendor work lacks a clear owner or next review point', 'Pilot change': 'Give vendor-dependent cases an accountable internal owner and an agreed review date', 'Owner': 'Vendor service manager', 'Success measure': 'Fewer unowned cases and shorter verified vendor-response intervals', 'Safeguard': 'Do not remove justified waiting periods or close cases merely to improve a score'}, {'Process issue to confirm': 'A recurring release pattern is followed by avoidable contacts', 'Pilot change': 'Pilot a readiness checklist and service-desk briefing for one comparable release family', 'Owner': 'Change manager', 'Success measure': 'Lower attributable post-release incident/contact demand and stable delivery quality', 'Safeguard': 'Validate the link and comparison window; retain emergency and risk-control routes'}])
display(process_improvements)

## 14. Full analysis journal and next iteration

The execution journal records what was actually loaded and assessed. The decision journal records the remaining business review and pilot decisions. Replace the pending notes after a stakeholder review; they do not imply that an organisation has approved a norm or that a pilot has occurred. If a case changes the explanation, return to the relevant PM² stage and rerun the same complete-case assessment.

Saved tables and figures may contain case identifiers or business context. Export is optional and disabled initially; keep shared results consistent with the source’s permitted use.

In [ ]:
decision_journal = pd.DataFrame([
    {"PM² stage": "Planning", "Decision": "Confirm the business goal, acceptable outcomes and policy owner", "Evidence to record": "Owner-approved scope and diagnostic cut points", "Status": "Pending business review"},
    {"PM² stage": "Extraction", "Decision": "Confirm source identity, observation boundary and case unit", "Evidence to record": "Execution journal and source identifiers", "Status": "Review computed journal"},
    {"PM² stage": "Data processing", "Decision": "Retain whole cases and review changing context or unavailable endpoints", "Evidence to record": "Context profiles and full-case witnesses", "Status": "Review computed evidence"},
    {"PM² stage": "Mining and analysis", "Decision": "Choose a lead that survives meaningful preference and threshold scenarios", "Evidence to record": "Coverage, signed drivers and shortlist sensitivity", "Status": "Review computed comparisons"},
    {"PM² stage": "Evaluation", "Decision": "Distinguish avoidable work, necessary control and recording gaps", "Evidence to record": "Source-case explanation and integrity checks", "Status": "Pending case interpretation"},
    {"PM² stage": "Process improvement and support", "Decision": "Approve one feasible pilot with a business outcome and stopping rule", "Evidence to record": "Owner, comparable cases, quality safeguards and follow-up date", "Status": "No pilot undertaken"},
])
display(journal(runs))
display(decision_journal)
if EXPORT_RESULTS:
    save_outputs(runs, Path("results") / "BPIC2014")